TREC DL 2020 pairwise LLM judge (30-query PoC)

Same protocol as `jusbrasil_soc_llms.ipynb`: pairwise comparisons, exponential NDCG, append to `data/PoC_results.csv`.

TREC Deep Learning is **not** one collection cut into 70/10/20. The official passage split is:

- **Train:** MS MARCO passage train (sparse binary labels, hundreds of thousands of queries)
- **Validation:** MS MARCO passage dev (`dev` / `dev.small`)
- **Test:** TREC DL 2020 — 54 NIST-judged queries, grades 0–3

This notebook evaluates only the **2020 test** set. TREC DL 2019 is a separate previous-year test set, not the 2020 validation split.

Official 2020 qrels have ~200 passages per query. The pilot samples 30 of the 54 test queries and keeps at most 18 docs each so the sort stays comparable to Jusbrasil `small_test`.

In [2]:
from pathlib import Path

import pandas as pd

Data loading

In [3]:
DATA_DIR = Path("/Users/david/Documents/dev/llm-as-judge-for-ltr/notebooks/data")
TEST_PATH = DATA_DIR / "trec_dl_2020_candidates_with_text.csv"

MAX_QUERIES = 30
MAX_DOCS_PER_QUERY = 18
RANDOM_STATE = 42
BODY_CHARS = 3000

# Official TREC DL 2020 test qrels + passage text. Train/val are MS MARCO, not this file.
trec_test = pd.read_csv(TEST_PATH)
print("TREC DL 2020 test rows:", len(trec_test))
print("Test queries:", trec_test["query_id"].nunique())
print(trec_test.groupby("query_id").size().describe())

TREC DL 2020 test rows: 11386
Test queries: 54
count     54.000000
mean     210.851852
std       56.358156
min      152.000000
25%      168.000000
50%      199.500000
75%      222.000000
max      368.000000
dtype: float64


Pilot subset from the 2020 **test** queries (30 of 54), Jusbrasil-sized slates

In [4]:
def sample_slate(group: pd.DataFrame, n: int, random_state: int) -> pd.DataFrame:
    if len(group) <= n:
        return group
    pos = group.loc[group["relevance_label"] >= 1]
    neg = group.loc[group["relevance_label"] == 0]
    n_pos = min(len(pos), max(n // 2, min(len(pos), n - min(6, len(neg)))))
    n_neg = min(len(neg), n - n_pos)
    if n_pos + n_neg < n and len(pos) > n_pos:
        n_pos = min(len(pos), n - n_neg)
    parts = []
    if n_pos:
        parts.append(pos.sample(n=n_pos, random_state=random_state))
    if n_neg:
        parts.append(neg.sample(n=n_neg, random_state=random_state))
    return pd.concat(parts, ignore_index=True)


query_index = (
    trec_test[["query_id", "query_text"]]
    .drop_duplicates("query_id")
    .sort_values("query_id")
    .reset_index(drop=True)
)
n_take = min(MAX_QUERIES, len(query_index))
pilot_queries = query_index.sample(n=n_take, random_state=RANDOM_STATE)

test_df = (
    trec_test.loc[trec_test["query_id"].isin(pilot_queries["query_id"])]
    .groupby("query_id", group_keys=False)
    .apply(lambda g: sample_slate(g, MAX_DOCS_PER_QUERY, RANDOM_STATE))
    .reset_index(drop=True)
)

print("Pilot queries:", test_df["query_id"].nunique())
if "year" in test_df.columns:
    print("Year mix:")
    print(test_df.drop_duplicates("query_id")["year"].value_counts().sort_index().to_string())
print("Docs per query:")
print(test_df.groupby("query_id").size().describe())
print("Relevance mix:")
print(test_df["relevance_label"].value_counts().sort_index().to_string())

Pilot queries: 30
Year mix:
year
2020    30
Docs per query:
count    30.0
mean     18.0
std       0.0
min      18.0
25%      18.0
50%      18.0
75%      18.0
max      18.0
dtype: float64
Relevance mix:
relevance_label
0    199
1    162
2     94
3     85


/var/folders/9f/4brf4bwj2jg5n11qsp30715c0000gn/T/ipykernel_13911/4274441553.py:30: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: sample_slate(g, MAX_DOCS_PER_QUERY, RANDOM_STATE))


In [5]:
def preprocess_llms(df):
    out = df.rename(
        columns={
            "query_text": "query",
            "passage_text": "body",
            "relevance_label": "relevance",
        }
    )
    keep = [c for c in ["query_id", "doc_id", "query", "body", "relevance", "year"] if c in out.columns]
    return out[keep]


test_df = preprocess_llms(test_df)
test_df["body"].astype(str).str.len().describe()

count    540.000000
mean     333.470370
std      137.490459
min       67.000000
25%      248.750000
50%      304.000000
75%      379.500000
max      923.000000
Name: body, dtype: float64

Evaluating by LLMs

In [8]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path("/Users/david/Documents/dev/llm-as-judge-for-ltr/.env"))

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

# client = OpenAI(
#     base_url="http://localhost:11434/v1",
#     api_key="ollama",
# )

In [9]:
# MODEL = "qwen/qwen-2.5-72b-instruct"
# MODEL = "openai/gpt-4o-mini"
#MODEL = "gemma2:9b"
MODEL = "google/gemma-4-26b-a4b-it"

In [10]:
def format_doc(row):
    body = str(row.get("body", ""))[:BODY_CHARS]
    return f"Text: {body}"


def compare_documents(doc_a, doc_b, query_text):
    system_prompt = (
        "You are an expert information retrieval judge. "
        "Compare two passages and decide which is more relevant to the search query."
    )
    user_prompt = f"""Query: {query_text}

Document A:
{format_doc(doc_a)}

Document B:
{format_doc(doc_b)}

Which passage is more relevant to the query? Output exactly 'A' if document A is more relevant, 'B' if document B is more relevant. Do not provide any explanation or extra text."""

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.0,
            max_tokens=2,
        )
        winner = (response.choices[0].message.content or "").strip().upper()
        if "A" in winner:
            return 1
        if "B" in winner:
            return -1
        return 0
    except Exception as e:
        print(f"Error comparing documents: {e}")
        return 0

In [11]:
from functools import cmp_to_key

from tqdm.auto import tqdm


def run_pairwise_pilot(df):
    results = []
    grouped = list(df.groupby("query_id"))

    for query_id, group in tqdm(grouped, total=len(grouped), desc="Ranking queries"):
        query_text = group["query"].iloc[0]
        docs = group.to_dict(orient="records")

        def custom_compare(a, b):
            return compare_documents(a, b, query_text)

        docs.sort(key=cmp_to_key(custom_compare), reverse=True)

        for rank, doc in enumerate(docs, start=1):
            doc["llm_rank"] = rank
            results.append(doc)

    return pd.DataFrame(results)

/Users/david/Library/Caches/pypoetry/virtualenvs/llm-as-judge-for-ltr-UctXQ_0n-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
df_llm_ranked = run_pairwise_pilot(test_df)
print("Ranking complete.")
df_llm_ranked.head()

Ranking queries: 100%|██████████| 30/30 [37:25<00:00, 74.84s/it]

Ranking complete.


,query_id,doc_id,query,body,relevance,year,llm_rank
0,23849,8010561,are naturalization records public information,Department of State >> Delaware Public Archive...,3,2020,1
1,23849,5340997,are naturalization records public information,How to obtain vital records (such as birth cer...,1,2020,2
2,23849,7845081,are naturalization records public information,Naturalization After 1906. When the INS was cr...,3,2020,3
3,23849,8010559,are naturalization records public information,"The 1900, 1910, 1920 and 1930 federal census s...",2,2020,4
4,23849,6688741,are naturalization records public information,Naturalization Records. Naturalization is the ...,1,2020,5


In [13]:
import numpy as np


def ndcg_at_k(labels_sorted_by_score, labels_ideal, k):
    k = min(k, len(labels_sorted_by_score))
    predicted = labels_sorted_by_score[:k]
    ideal = np.sort(labels_ideal)[::-1][:k]
    discounts = np.log2(np.arange(1, k + 1) + 1)
    dcg = np.sum((np.power(2.0, predicted) - 1) / discounts)
    idcg = np.sum((np.power(2.0, ideal) - 1) / discounts)
    return 0.0 if idcg == 0.0 else dcg / idcg


ks = [1, 3, 5, 10]
rows = []
for query_id, g in df_llm_ranked.groupby("query_id"):
    y_q = g["relevance"].to_numpy(dtype=float)
    y_ranked = y_q[np.argsort(g["llm_rank"].to_numpy())]
    row = {"query_id": query_id, "num_docs": len(g)}
    for k in ks:
        row[f"ndcg@{k}"] = ndcg_at_k(y_ranked, y_q, k)
    rows.append(row)

metrics_llm = pd.DataFrame(rows)
for k in ks:
    print(f"LLM  NDCG@{k:<3} = {metrics_llm[f'ndcg@{k}'].mean():.4f}")

LLM  NDCG@1   = 0.8492
LLM  NDCG@3   = 0.8026
LLM  NDCG@5   = 0.8306
LLM  NDCG@10  = 0.8811


In [16]:
RESULTS_PATH = Path("/Users/david/Documents/dev/llm-as-judge-for-ltr/notebooks/data/PoC_result_trec2020.csv")


def append_poc_results(model, metrics_df, notes=""):
    row = {
        "model": model,
        "ndcg@1": round(metrics_df["ndcg@1"].mean(), 4),
        "ndcg@3": round(metrics_df["ndcg@3"].mean(), 4),
        "ndcg@5": round(metrics_df["ndcg@5"].mean(), 4),
        "ndcg@10": round(metrics_df["ndcg@10"].mean(), 4),
        "n_queries": len(metrics_df),
        "notes": notes,
    }
    new = pd.DataFrame([row])
    if RESULTS_PATH.exists():
        out = pd.concat([pd.read_csv(RESULTS_PATH), new], ignore_index=True)
    else:
        out = new
    out.to_csv(RESULTS_PATH, index=False)
    return out

In [17]:
append_poc_results(
    MODEL,
    metrics_llm,
    notes=f"pairwise LLM judge on TREC-DL 2020 test {test_df['query_id'].nunique()}q {BODY_CHARS} body size",
)

,model,ndcg@1,ndcg@3,ndcg@5,ndcg@10,n_queries,notes
0,google/gemma-4-26b-a4b-it,0.8492,0.8026,0.8306,0.8811,30,pairwise LLM judge on TREC-DL 2020 test 30q 30...
